# 1. Data acquisition 

In this section we will parse the Protein Data Bank (PDB) for kinase structures and download them into a dataset.

## Table of contents

Our data acquisition pipeline is subdivided in the following sections: 
1. [Data acquisition](#1)   
    1.1. [The data](#11)   
    1.2. [Parsing the PDB](#12)   
    1.3. [Data download](#13)
    1.4. [KLIFS dataset] (#14)


To get started, let's load some packages!

In [ ]:
import warnings

warnings.filterwarnings("ignore")

# File and system operations
import os
import sys
import subprocess
from glob import glob
import pickle
import shutil

# Data processing
import pandas as pd
import numpy as np
import mdtraj as md

# Network and parallel processing
import requests
import time
import multiprocessing
import concurrent.futures

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# custom utility functions and class
from workflow.utilities import count_pdb_files, braf_res, clear_and_make, make_seg, copy_filtered_pdbs, copy_motif_filtered_datasets
from workflow.utilities import PDBDownloader

## 1.1 The data <a id="11"></a>
Our approach involves searching for protein homologs to the reference sequence: a BRAF kinase (PDB code: 6UAN).

BRAF is a key part of the MAPK/ERK pathway. This pathway relays signals from outside the cell, like growth factors binding to receptor tyrosine kinases, to control cell growth, division, survival, and differentiation.

The active site sits in a cleft between the small N‑terminal lobe and larger C‑terminal lobe of the kinase domain. 
ATP binds in a pocket on the N‑lobe side of the kinase domain, at the P-binding loop. 
The protein substrate, mainly MEK kinase, contacts a broad surface on the C‑lobe of BRAF. 
Activation loop and αC helix interact with residues in the binding site, regulating activation. 

![State of the workflow](images/BRAFSlide1.png)

When creating our dataset, we take as structural reference the BRAF structure since we are familiar with its typical regulatory role during phosphorilation. We query the InterPro database online at https://www.ebi.ac.uk/interpro/ to find structures in the PDB that match the protein kinase-like domain family. InterPro is a database that classifies protein sequences into families and predicts the presence of domains and important sites.

Our query input is the BRAF sequence and we filter for structures that are part of the "Protein kinase-like domain superfamily" (IPR011009) and that are included in the PDB.

## 1.2 Parsing the PDB <a id="12"></a>
Our approach involves searching for protein homologs to the reference sequence: a BRAF kinase (PDB code: 6UAN).

We query the InterPro REST API live to retrieve all PDB structures annotated with protein kinase-like domain 
superfamily **IPR011009**.  
Results are cached to a local TSV file on first run; delete the file to force a fresh fetch. This ensures that the 
downloaded dataset is sensitive to changes of InterPro database.

Let's start by writing all PDB codes to a list.

In [ ]:
downloader = PDBDownloader()
pdb_data = downloader.fetch_interpro_structures(
    entry_id="IPR011009",
    cache_path=None, 
)
pdb_ids = pdb_data['Accession'].tolist()
print(f"Found {len(pdb_ids)} structures for IPR011009")

## 1.3 Data download <a id="13"></a>
Here we download the structures output from the InterPro query.

The same `PDBDownloader` instance carries out multi-threaded PDB/CIF download using up to 2× CPU cores. We now download the structures returned by the InterPro query.

In [ ]:
downloader = PDBDownloader()
downloader.parallel_download(pdb_ids, "Results/InterPro_PDBs")

We can now check how many structures from the InterPro query were actually downloaded.

In [ ]:
folder_path = "Results/InterPro_PDBs"
file_names = [
    os.path.splitext(f)[0]
    for f in os.listdir(folder_path)
    if os.path.isfile(os.path.join(folder_path, f)) and f.lower().endswith((".pdb", ".cif"))
]
pdb_raw = pd.DataFrame({"PDBs": file_names})

pdb_data['Downloaded'] = pdb_data['Accession'].str.upper().isin(pdb_raw['PDBs']).map({True: True, False: False})

counts = pdb_data['Downloaded'].value_counts().to_dict()
print(f"Downloaded: {counts.get(True, 0)}, Failed: {counts.get(False, 0)}")

# Save for standalone chain-extraction cells (read with same path below)
os.makedirs("Results", exist_ok=True)
pdb_data_for_extraction_path = "Results/pdb_data_for_chain_extraction.tsv"
pdb_data.to_csv(pdb_data_for_extraction_path, sep="\t", index=False)

We can save the names of failed PDB downloads for future reference.

In [ ]:
fail_list = pdb_data[pdb_data['Downloaded'] == False]
if not fail_list.empty:
    fail_list.to_csv('fail_list.csv')

## 1.4 KLIFS dataset  <a id="14"></a>
We compare **downloaded InterPro structures** (unique PDB IDs in `Results/InterPro_PDBs` from §1.3) against the **KLIFS** database (https://klifs.net/api/) at the **structure** level — not per chain.

Call **`klifs.build_full_klifs_catalog()`** once (order of **~1100** `structures_list` requests, a few minutes; then cached). It walks every `kinase_ID` from `/kinase_information`, skips kinases that return HTTP 400 (no structures), and writes:

* `Results/KLIFS/klifs_full_catalog_pdb_chain.tsv` — unique KLIFS `(PDB, chain)` rows (full API sweep; we take unique PDB IDs for the Venn)
* `Results/KLIFS/klifs_full_catalog_summary.json` — row counts and timing

Then **`klifs.build_structure_inventory("Results/InterPro_PDBs")`** marks each downloaded PDB ID as present in KLIFS or not, and **`klifs.plot_venn_structures()`** draws the Venn so region **01** = KLIFS catalogue PDB IDs minus your download. If you skip `build_full_klifs_catalog()`, membership falls back to `structures_pdb_list` for the downloaded IDs only (KLIFS-only crescent will be incomplete).

Outputs:

* `Results/KLIFS/klifs_structure_inventory.tsv` — per-PDB `in_klifs` flags
* `Results/KLIFS/klifs_structure_venn_summary.json` — overlap counts
* `Results/KLIFS/venn_interpro_vs_klifs_structures.png` — saved figure

Run `build_structure_inventory(..., force=True)` to refresh after new downloads; run `build_full_klifs_catalog(force=True)` to refresh the full catalogue after KLIFS updates.

In [ ]:
from workflow.klifs_filter import KLIFSOverlap

klifs = KLIFSOverlap()
klifs.build_full_klifs_catalog()  # ~1100 API calls first time (~minutes); then cached
inventory = klifs.build_structure_inventory("Results/InterPro_PDBs")  # cached on subsequent runs
klifs.plot_venn_structures()
